# Meeting minutes creator

In this colab, we make a meeting minutes program.

It includes useful code to connect your Google Drive to your colab.

Upload your own audio to make this work!!

https://colab.research.google.com/drive/1KSMxOCprsl1QRpt_Rq0UqCAyMtPqDQYx?usp=sharing

This should run nicely on a low-cost or free T4 box.

### BUT FIRST - Something cool - really showing you how "model inference" works via OpenAI

In [1]:
from openai import OpenAI
import os

In [2]:
# from visualizer import TokenPredictor, create_token_graph, visualize_predictions

# message = "In one sentence, describe the color orange to someone who has never been able to see"
# # model_name = "gpt-4.1-mini"
# model_name = "gpt-4o-mini"

# predictor = TokenPredictor(model_name)
# predictions = predictor.predict_tokens(message)
# G = create_token_graph(model_name, predictions)
# plt = visualize_predictions(G)
# plt.show()

In [17]:
import os
from openai import OpenAI

class TokenPredictor:
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ["OPENROUTER_API_KEY"],
        )

    def predict_tokens(self, prompt: str, max_tokens: int = 100, top_k: int = 3):
        # NOTE: OpenRouter often does NOT provide streamed logprobs.
        # Use non-streaming for reliable logprobs.
        resp = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0,
            seed=42,
            logprobs=True,
            top_logprobs=top_k,
            stream=False,
        )

        choice = resp.choices[0]
        full_text = choice.message.content or ""

        token_predictions = []
        lp = getattr(choice, "logprobs", None)
        if lp and getattr(lp, "content", None):
            for item in lp.content:
                # item.token, item.logprob, item.top_logprobs (list)
                token_predictions.append({
                    "token": item.token,
                    "logprob": item.logprob,
                    "top_logprobs": (
                        [{"token": t.token, "logprob": t.logprob} for t in (item.top_logprobs or [])]
                    ),
                })

        return token_predictions, resp


In [18]:
message = "In one sentence, describe the color orange to someone who has never been able to see"
model_name = "openai/gpt-4.1-mini"

predictor = TokenPredictor(model_name)
predictions, resp = predictor.predict_tokens(message)

In [26]:
resp.choices[0].logprobs

In [20]:
resp.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Orange is the warm, bright feeling you get from the heat of a gentle sunset or the sweet, tangy taste of a ripe citrus fruit.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='completed')

In [38]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],)

In [40]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": message}],
    max_tokens=100,
    temperature=0,
    logprobs=True,
    top_logprobs=3,
    seed=42,
    stream=True,
)